# Occlusion-aware CSCI fingerprints (MEVID-SC)

**Why:** CSCI averages its features over *every* frame of a clip. Frames where the person is hidden, or where the box drifted onto grass or a wall, pollute the fingerprint. The ClipQuality study found that 76% of the bad clips still contain the person in some frames.

**Method (no labels, no training, CSCI weights and code unchanged):**
1. **Check frames:** a pose model checks every 4th frame of every clip. Each frame in between takes the result of the nearest checked frame.
2. **Keep only frames where the person is clearly visible**, and rebuild each clip from those frames:
   - `visible` variant (main): the person is found and at least 50% of the body is visible;
   - `found` variant: the person is found.
3. **Too few good frames:** a clip with fewer than 8 good frames keeps all its frames, so it is unchanged.
4. **Rerun CSCI** on the filtered clips. Same clips, same order, same queries.

**Fair test:**
- Original CSCI is compared with occlusion-aware CSCI on the same clips.
- Accuracy: mAP and Rank-1.
- Consistent ID: the cut is calibrated on one group of people and tested on the other, over 50 tests. Every clip counts, so nothing is set aside as unresolved.
- Changed clips: do they now find the right person?

**Needs an A100.**

| Step | Time |
|---|---|
| Unpack frames | about 10 min |
| Frame check | about 15 min |
| CSCI, `visible` variant | about 45 to 60 min |
| CSCI, `found` variant | about 45 to 60 min (optional: stop after `visible` and run cell 6) |

Everything is saved to Drive, and reruns skip finished parts.

In [ ]:
# 1. Drive, paths, settings (frozen before any result), and unpack the
#    MEVID-SC test frames to local disk (all frames, about 10 minutes).
import os, glob, json, time, subprocess, sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats, required=True):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    if required:
        raise FileNotFoundError(pats[0])

SC_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
                               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
CS = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
                           'test_templates_csci_sc.npz',
                           f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz']))
WT = find([f'{SD}/Data/MEVID-results/CSCI/ez_eva02_vid_hybrid_extra_best.pth',
           f'{SD}/**/CSCI/ez_eva02_vid_hybrid_extra_best.pth'])
OUT = os.path.join(os.path.dirname(CS), 'OcclusionAware-SameClothes')
os.makedirs(OUT, exist_ok=True)

SETTINGS = {
    'check_every': 4,            # person check on every 4th frame; the
                                 # frames in between take the nearest check
    'pose_model': 'yolo11m-pose.pt', 'imgsz': 320, 'det_conf': 0.25,
    'keypoint_conf': 0.5,
    # a frame is KEPT if the person is found and at least this share of
    # the 17 body keypoints is visible (occlusion-aware); the second
    # variant only requires that the person is found
    'variants': {'visible': 0.5, 'found': 0.0},
    'min_kept_frames': 8,        # CSCI's clip length; below this the clip
                                 # keeps all its frames (unchanged)
    'precision_target': 0.90, 'n_splits': 25, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)

A = f'{SC_DIR}/annotations'
names = [l.strip() for l in open(f'{A}/test_name.txt') if l.strip()]
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
qidx = np.loadtxt(f'{A}/query_IDX.txt').astype(np.int64)
pid, cam = info[:, 2], info[:, 4]
LOCAL = '/content/sc_test'
if not os.path.isdir(f'{LOCAL}/bbox_test'):
    os.makedirs(LOCAL, exist_ok=True)
    t0 = time.time()
    print('Unpacking the MEVID-SC test frames ...')
    subprocess.run(['tar', '-xf', find([f'{SC_DIR}/mevid-sc-bbox-test.tar']),
                    '-C', LOCAL], check=True)
    print(f'done in {time.time() - t0:.0f} s')
n_img = sum(len(f) for _, _, f in os.walk(f'{LOCAL}/bbox_test'))
print(f'{len(info)} clips, {len(np.unique(pid))} people, {len(names):,} '
      f'frames listed, {n_img:,} on disk | results -> {OUT}')
assert n_img >= len(names)


In [ ]:
# 2. Person check on every 4th frame of every clip (pose model, no
#    labels). Saved to Drive in chunks, so a restart resumes.
#    About 240,000 images; about 10 to 20 minutes on an A100.
!pip -q install ultralytics
import cv2, torch
from ultralytics import YOLO
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
st = SETTINGS
check = []
for s, e in info[:, :2]:
    c = list(range(s, e + 1, st['check_every']))
    if c[-1] != e:
        c.append(e)
    check += c
check = np.array(check)
path = lambda k: f'{LOCAL}/bbox_test/{names[k][:4]}/{names[k]}'


class Imgs(Dataset):
    def __init__(self, ks): self.ks = ks
    def __len__(self): return len(self.ks)
    def __getitem__(self, i): return cv2.imread(path(self.ks[i]))


pose = YOLO(st['pose_model'])
CH = 20000
for c0 in range(0, len(check), CH):
    fn = f'{OUT}/personcheck_{c0 // CH:03d}.npz'
    if os.path.exists(fn):
        continue
    ks = check[c0:c0 + CH]
    found = np.zeros(len(ks), np.int8); vis = np.zeros(len(ks), np.float32)
    dl = DataLoader(Imgs(ks), batch_size=128, num_workers=8,
                    collate_fn=lambda b: b)
    i = 0
    for batch in tqdm(dl, desc=f'chunk {c0 // CH + 1}/'
                                f'{-(-len(check) // CH)}', leave=False):
        res = pose.predict(batch, imgsz=st['imgsz'], conf=st['det_conf'],
                           verbose=False, half=True)
        for im, r in zip(batch, res):
            if len(r.boxes):
                H, W = im.shape[:2]
                xy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
                area = (xy[:, 2] - xy[:, 0]) * (xy[:, 3] - xy[:, 1]) / (H * W)
                m = int(np.argmax(area * cf))
                kp = r.keypoints.data.cpu().numpy()[m]
                found[i] = 1; vis[i] = (kp[:, 2] >= st['keypoint_conf']).mean()
            i += 1
    np.savez(fn, k=ks, found=found, vis=vis)
parts = [np.load(f) for f in sorted(glob.glob(f'{OUT}/personcheck_*.npz'))]
K = np.concatenate([p['k'] for p in parts])
assert len(K) == len(check), 'person check incomplete; rerun this cell'
FOUND = np.concatenate([p['found'] for p in parts])
VIS = np.concatenate([p['vis'] for p in parts])
print(f'{len(K):,} frames checked; person found in {100 * FOUND.mean():.1f}%, '
      f'median visible body share {np.median(VIS[FOUND == 1]):.2f}')


In [ ]:
# 3. Build the filtered clips: same clips, same order, same people and
#    cameras; only frames without a (visible) person are removed.
#    Every frame takes the result of the nearest checked frame.
o = np.argsort(K); K, FOUND, VIS = K[o], FOUND[o], VIS[o]
REPORT = []
for v, thr in st['variants'].items():
    new_names, new_info, changed, fallback = [], [], [], []
    for t, (s, e, p, oid, c) in enumerate(info):
        fr = np.arange(s, e + 1)
        lo, hi = np.searchsorted(K, s), np.searchsorted(K, e, 'right')
        kk = K[lo:hi]
        near = lo + np.abs(fr[:, None] - kk[None, :]).argmin(1) \
            if len(fr) * len(kk) < 4e6 else \
            lo + np.clip(np.searchsorted(kk, fr), 0, len(kk) - 1)
        keep = (FOUND[near] == 1) & (VIS[near] >= thr)
        if keep.sum() < st['min_kept_frames']:
            keep[:] = True; fallback.append(t)
        elif not keep.all():
            changed.append(t)
        start = len(new_names)
        new_names += [names[k] for k in fr[keep]]
        new_info.append([start, len(new_names) - 1, p, oid, c])
    root = f'/content/root_{v}'
    os.makedirs(root, exist_ok=True)
    with open(f'{root}/test_name.txt', 'w') as f:
        f.write('\n'.join(new_names) + '\n')
    np.savetxt(f'{root}/track_test_info.txt', np.array(new_info), fmt='%d')
    np.savetxt(f'{root}/query_IDX.txt', qidx, fmt='%d')
    for k, src in {'train_name.txt': f'{A}/train_name.txt',
                   'track_train_info.txt': f'{A}/track_train_info.txt',
                   'bbox_test': f'{LOCAL}/bbox_test'}.items():
        if not os.path.exists(f'{root}/{k}'):
            os.symlink(src, f'{root}/{k}')
    np.savez(f'{OUT}/filter_{v}.npz', changed=np.array(changed, int),
             fallback=np.array(fallback, int),
             kept=np.diff(np.array(new_info)[:, :2], axis=1)[:, 0] + 1)
    REPORT.append({'variant': v, 'keep rule': (f'person found, >= {thr:.0%} '
                   'of body visible' if thr else 'person found'),
                   'frames kept %': round(100 * len(new_names) / len(names), 1),
                   'clips changed': len(changed),
                   'clips unchanged (too few good frames)': len(fallback)})
display(pd.DataFrame(REPORT).set_index('variant'))


In [ ]:
# 4. CSCI: authors' code and released MEVID weights, unchanged, with the
#    same two patches as all earlier runs (feature dump; skip loading a
#    training clip).
REPO = '/content/ICCV-CSCI-Person-ReID'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ppriyank/'
                    'ICCV-CSCI-Person-ReID.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.9', 'einops', 'yacs', 'mmengine', 'decord',
                'tensorboardX', 'tensorboard', 'termcolor', 'tabulate',
                'h5py', 'scikit-learn', 'lmdb', 'albumentations',
                'torchinfo', 'grad-cam', 'opencv-python-headless'],
               check=True)
f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
anchor = '    distmat = distmat.cpu()\n'
hook = (anchor +
        "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
        "        import numpy as _np\n"
        "        _np.savez(os.environ['CSCI_DUMP'], "
        "qf=qf.float().cpu().numpy(), gf=gf.float().cpu().numpy(), "
        "q_pids=q_pids.numpy(), g_pids=g_pids.numpy(), "
        "q_camids=q_camids.numpy(), g_camids=g_camids.numpy())\n")
if 'CSCI_DUMP' not in src:
    assert src.count(anchor) == 1, 'eval_mevid.py changed upstream'
    src = src.replace(anchor, hook)
    if '\nimport os' not in src:
        src = 'import os\n' + src
    open(f, 'w').write(src)
f = f'{REPO}/data/dataset_loader.py'; src = open(f).read()
old_l = ('        self.return_index = return_index\n'
         '        self.__getitem__(0)\n')
new_l = ('        self.return_index = return_index\n'
         '        if not train:\n            self.__getitem__(0)\n')
if old_l in src:
    open(f, 'w').write(src.replace(old_l, new_l))
print('CSCI ready; weights:', WT)


In [ ]:
# 5. Run CSCI on the filtered clips (about 45 to 60 minutes per variant
#    on an A100). The 'visible' variant runs first; you may stop after it.
gidx = np.setdiff1d(np.arange(len(info)), qidx)
for v in st['variants']:
    OUTF = f'{OUT}/test_templates_csci_{v}.npz'
    if os.path.exists(OUTF):
        print('done already:', OUTF); continue
    DUMP = f'{OUT}/csci_dump_{v}.npz'
    if not os.path.exists(DUMP):
        env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO)
        cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12359',
               'train_two_step.py', '--eval', '--resume',
               '--config_file', 'configs/mevid_eva02_l_cloth.yml',
               'DATA.ROOT', f'/content/root_{v}/', 'TEST.WEIGHT', WT,
               'SOLVER.SEED', '12345', 'TRAIN.TRAIN_VIDEO', 'True',
               'DATA.DATASET', 'mevid', 'MODEL.NAME',
               'ez_eva02_vid_hybrid_extra', 'TRAIN.COLOR_PROFILE', '17',
               'DATA.TEST_BATCH', '32']
        t0 = time.time()
        with open(f'{OUT}/csci_log_{v}.txt', 'w') as log:
            p = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                                 stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT)
            for line in p.stdout:
                log.write(line)
                if any(k in line for k in ['top1', 'mAP', 'Error',
                                           'Traceback', 'Extract']):
                    print(line.rstrip())
            p.wait()
        print(f'{v}: exit {p.returncode} after {(time.time() - t0) / 60:.0f} '
              'min (non-zero after the dump is CSCI\'s optional extra '
              'protocol; harmless)')
    assert os.path.exists(DUMP), f'CSCI stopped before dumping ({v}); see log'
    z = np.load(DUMP)
    assert (z['q_pids'] == pid[qidx]).all() and (z['g_pids'] == pid[gidx]).all()
    T = np.zeros((len(info), z['qf'].shape[1]), np.float32)
    T[qidx], T[gidx] = z['qf'], z['gf']
    T /= np.maximum(np.linalg.norm(T, axis=1, keepdims=True), 1e-8)
    np.savez(OUTF, templates=T, pid=pid, cam=cam)
    print('saved', OUTF)


In [ ]:
# 6. Results: original CSCI (all frames) vs occlusion-aware CSCI.
#    (a) accuracy (mAP, Rank-1); (b) consistent ID across cameras, the
#    cut calibrated on one group of people and tested on the other, 25
#    random splits x 2 directions = 50 tests, no gate, every clip counts;
#    (c) the clips that changed: does the right person get closer?
from scipy.cluster.hierarchy import linkage, fcluster
from tqdm.auto import tqdm
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


gidx = np.setdiff1d(np.arange(len(info)), qidx)
te = np.load(f'{CS}/test_templates_csci_sc.npz')
assert (te['pid'] == pid).all()
FEATS = {'CSCI (all frames)': unit(te['templates'])}
for v in st['variants']:
    fn = f'{OUT}/test_templates_csci_{v}.npz'
    if os.path.exists(fn):
        FEATS[f'CSCI occlusion-aware ({v})'] = unit(np.load(fn)['templates'])


def ap_cmc(order, good, junk):
    order = order[~np.isin(order, junk)]
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc


def accuracy(X):
    S = X[qidx] @ X[gidx].T
    aps, r1 = [], []
    for i, q in enumerate(qidx):
        sp = np.flatnonzero(pid[gidx] == pid[q])
        sc = np.flatnonzero(cam[gidx] == cam[q])
        good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
        if good.size:
            a, c = ap_cmc(np.argsort(-S[i]), good, junk)
            aps.append(a); r1.append(c[0])
    return {'mAP %': 100 * np.mean(aps), 'Rank-1 %': 100 * np.mean(r1)}


def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0); tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}


def tree(X, idx):
    D = np.clip(1 - X[idx] @ X[idx].T, 0, None); D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    return linkage(squareform(D, checks=False), 'average')


def calibrate(L, idx):
    best = 0.01
    for t in np.linspace(0.01, 1.5, 150):
        if score(fcluster(L, t, 'distance'), pid[idx], cam[idx]
                 )['pair precision %'] >= 100 * st['precision_target']:
            best = t
        else:
            break
    return best


ACC = pd.DataFrame({n: accuracy(X) for n, X in FEATS.items()}).T.round(2)
print('(a) Accuracy on MEVID-SC (same clothes, cross-camera queries)')
display(ACC)
ACC.to_csv(f'{OUT}/accuracy.csv')

rng = np.random.default_rng(st['seed'])
people = np.unique(pid)
ROWS = []
for s in tqdm(range(st['n_splits']), desc='50 tests'):
    perm = rng.permutation(people)
    a = np.isin(pid, perm[:len(people) // 2])
    G = [np.flatnonzero(a), np.flatnonzero(~a)]
    for n, X in FEATS.items():
        Ls = [tree(X, g) for g in G]
        for c, t in [(0, 1), (1, 0)]:
            cut = calibrate(Ls[c], G[c])
            r = score(fcluster(Ls[t], cut, 'distance'), pid[G[t]], cam[G[t]])
            r.update(features=n, split=s, direction=c)
            ROWS.append(r)
REP = pd.DataFrame(ROWS)
REP.to_csv(f'{OUT}/consistent_id_50tests.csv', index=False)
COLS = ['IDF1 %', 'IDs per person', 'pair precision %', 'cross-camera recall %']
print('\n(b) Consistent ID, mean (sd) over 50 tests, every clip counts:')
display(REP.groupby('features', sort=False)[COLS].agg(['mean', 'std']).round(2))
base = REP[REP.features == 'CSCI (all frames)'].set_index(['split', 'direction'])
PAIR = []
for n in list(FEATS)[1:]:
    o = REP[REP.features == n].set_index(['split', 'direction'])
    for m, better in [('IDF1 %', 1), ('IDs per person', -1),
                      ('pair precision %', 1), ('cross-camera recall %', 1)]:
        d = o[m] - base[m]
        PAIR.append({'occlusion-aware variant': n, 'metric': m,
                     'mean change': round(d.mean(), 2),
                     '95% range': f'[{np.percentile(d, 2.5):.2f}, '
                                  f'{np.percentile(d, 97.5):.2f}]',
                     'better in % of tests': round(100 * (better * d > 0).mean())})
PAIR = pd.DataFrame(PAIR)
PAIR.to_csv(f'{OUT}/paired_vs_original.csv', index=False)
print('\nPaired change vs original CSCI (50 tests):')
display(PAIR)


def margins(X, idx):
    """for each clip: best same-person other-camera similarity minus best
    other-person similarity (positive = the right person is closest)"""
    S = X[idx] @ X.T
    same = pid[None, :] == pid[idx][:, None]
    other_cam = cam[None, :] != cam[idx][:, None]
    pos = np.where(same & other_cam, S, -np.inf).max(1)
    neg = np.where(~same, S, -np.inf).max(1)
    ok = np.isfinite(pos)
    return (pos - neg)[ok]


DIAG = []
for v in st['variants']:
    n = f'CSCI occlusion-aware ({v})'
    if n not in FEATS:
        continue
    fz = np.load(f'{OUT}/filter_{v}.npz')
    for lab, idx in [('clips that changed', fz['changed']),
                     ('clips unchanged', np.setdiff1d(np.arange(len(pid)),
                                                      fz['changed']))]:
        if len(idx) == 0:
            continue
        b, aft = margins(FEATS['CSCI (all frames)'], idx), margins(FEATS[n], idx)
        DIAG.append({'variant': v, 'clips': lab, 'n': len(b),
                     'right person closest, before %': round(100 * (b > 0).mean(), 1),
                     'after %': round(100 * (aft > 0).mean(), 1),
                     'mean margin before': round(b.mean(), 3),
                     'after': round(aft.mean(), 3)})
print('\n(c) Do the changed clips now find the right person?')
display(pd.DataFrame(DIAG))
pd.DataFrame(DIAG).to_csv(f'{OUT}/changed_clips.csv', index=False)
import shutil
shutil.make_archive('/content/OcclusionAware_results', 'zip', OUT,
                    )
print('Saved to', OUT, '\nZip: /content/OcclusionAware_results.zip')
